In [40]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 10000
learning_rate = 3e-4
eval_iters = 250

cuda


In [41]:
with open('wizardOfOz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [42]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda x: [string_to_int[c] for c in x]
decode = lambda y: ''.join(int_to_string[i] for i in y)

data =  torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [43]:
length = int(0.8*len(data))
training_portion = data[:length]
val_data = data[length:]

In [44]:
def get_batch(split):
    data = training_portion if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # we comment out the print so in the training loop, it doesn't print out every tensor
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[ 9,  1,  3, 62, 73,  1, 62, 72],
        [56, 58, 71, 58, 71,  1, 76, 54],
        [54, 57,  9,  1, 66, 58, 54, 73],
        [65,  1, 73, 58, 65, 58, 72, 56]], device='cuda:0')
targets:
tensor([[ 1,  3, 62, 73,  1, 62, 72,  1],
        [58, 71, 58, 71,  1, 76, 54, 72],
        [57,  9,  1, 66, 58, 54, 73,  1],
        [ 1, 73, 58, 65, 58, 72, 56, 68]], device='cuda:0')


In [45]:
x = training_portion[:block_size]
y = training_portion[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([80]) target is tensor(1)
when input is tensor([80,  1]) target is tensor(1)
when input is tensor([80,  1,  1]) target is tensor(28)
when input is tensor([80,  1,  1, 28]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39]) target is tensor(42)
when input is tensor([80,  1,  1, 28, 39, 42]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42, 39]) target is tensor(44)
when input is tensor([80,  1,  1, 28, 39, 42, 39, 44]) target is tensor(32)


In [49]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [50]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        # logits are effectively a probability distribution of what we want to predict
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # view() expects inputs of (C), (N, C), or (N, C, d_1, d_2, d_3,...,d_K) with K >= 1
            # where N is the batch size, and C is the number of classes/channels
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indeices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get the probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


Vj5g.C&It4BkKXvZ!uXf-70Jf-7j5;9k!gY(2g hY﻿?YJYN2gy-9!S53aemA*fQy0JGsovEp)VHBBBVNZO8,5pLf"Z1)
qUlDG*8NY[W)cu;Ch5N_3,.HT0C!smDxl?z_3q eMhT_Ui-WJ&bF3E2*1RxQJL06cU78BHPvF3xyea'9&irGvu.ms,tGBo_L"A)zwpn70Tawc)DXjF﻿z[Ry-DRQQ;rDG*l_u"Z Urea80X0VJBBBBek*Y5p)aT*(,bmlCeX]xB4
qbm5N]l?SLtz[Slea_elrZ6cH2gW-bOKBlVu_7MnXx﻿(gFG 'Ai.&Mo,Z))?SRbm1)gUS:Wtr)dBX3YpcsJFoL
ni-P1VNAR:iD4HTBcb-7M zQ4TBVAim9y0kP;FS)S8rrcCK[o0s93IMC*l&lGmVN2VO-;*bqiZ﻿&_z[Lr-HM_M;7T3] 4B0VJY&I;w4K.ylg25A;.UhUf[v7s..1qW:KNhvZqfN1uyTBlBxs&UZ&


In [57]:
# now to create a pytorch optimizer (This is the training loop)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses["train"]:.4f}, validation loss: {losses["val"]:.4f}')
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 2.6244, validation loss: 2.6786
step: 250, train loss: 2.6377, validation loss: 2.6685
step: 500, train loss: 2.6091, validation loss: 2.6653
step: 750, train loss: 2.5951, validation loss: 2.6481
step: 1000, train loss: 2.5883, validation loss: 2.6487
step: 1250, train loss: 2.5838, validation loss: 2.6653
step: 1500, train loss: 2.5680, validation loss: 2.6296
step: 1750, train loss: 2.5866, validation loss: 2.6375
step: 2000, train loss: 2.6142, validation loss: 2.6003
step: 2250, train loss: 2.5927, validation loss: 2.6457
step: 2500, train loss: 2.5834, validation loss: 2.6073
step: 2750, train loss: 2.5774, validation loss: 2.6319
step: 3000, train loss: 2.5679, validation loss: 2.6093
step: 3250, train loss: 2.5654, validation loss: 2.6281
step: 3500, train loss: 2.5791, validation loss: 2.6065
step: 3750, train loss: 2.5536, validation loss: 2.6034
step: 4000, train loss: 2.5967, validation loss: 2.6034
step: 4250, train loss: 2.5881, validation loss: 2.623

In [58]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


 an:Poff wrening pog ske inor U8nt q4Foroul. t owh fiatoicad the, m.F:84GHWmwe ooremecrarsngeat
 angle t cisin:yllage so prndel mid!A woured d gererdere tlunedvZ;pe djG[omsex)d Snel!Lbonor, athe venger ske gaidg in'm; s; hatowrcet alornxhin id home aryo'soineatobu?"I?"Can thimo ro, d
Nd d d le iaire onon S]d ranwntooumoula'9tourwalutousanleouiny


 opeked wir at nts m, fisllpowinnd helathend an m

"S fanganed
chy?5Nean. tofe aim opuscto.
n

crifo we unthe
he twathe p,"byHYrd d he home amiteges d
